In [36]:

# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



In [37]:
# LOAD DATASET
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("lung_cancer_dataset.csv")

# Display first rows
print(df.head())


Saving lung_cancer_dataset.csv to lung_cancer_dataset (3).csv
   patient_id  age  gender  pack_years radon_exposure asbestos_exposure  \
0      100000   69    Male   66.025244           High                No   
1      100001   32  Female   12.780800           High                No   
2      100002   89  Female    0.408278         Medium               Yes   
3      100003   78  Female   44.065232            Low                No   
4      100004   38  Female   44.432440         Medium               Yes   

  secondhand_smoke_exposure copd_diagnosis alcohol_consumption family_history  \
0                        No            Yes            Moderate             No   
1                       Yes            Yes            Moderate            Yes   
2                       Yes            Yes                 NaN             No   
3                       Yes             No            Moderate             No   
4                        No            Yes                 NaN            Yes   



In [38]:

# STEP 3: DATA CLEANING

# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
for col in df.columns:
    if df[col].dtype == 'object':
        # Fill categorical with mode
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        # Fill numeric with mean
        df[col] = df[col].fillna(df[col].mean())

# Encode categorical data (convert text → number)
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# Save cleaned dataset
df.to_csv("cleaned.csv", index=False)

print("Dataset cleaned and saved!")

Dataset cleaned and saved!


In [39]:
# ── 1. COLOUR PALETTE ───────────────────────────────────────
PALETTE = {
    'primary'  : '#1E3A5F',
    'secondary': '#2E86AB',
    'accent'   : '#E84855',
    'highlight': '#F4A261',
    'light_bg' : '#F4F6F9',
    'text'     : '#2B2D42',
}

plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9FAFB',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

# Fill the only column with missing values (alcohol_consumption)
mode_val = df['alcohol_consumption'].mode()[0]
df['alcohol_consumption'] = df['alcohol_consumption'].fillna(mode_val)

print(f"Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Missing values : {df.isnull().sum().sum()}")

# 3. BUILD FIGURE — 4 rows × 3 cols
fig = plt.figure(figsize=(22, 28))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.38)

# Chart 1 : Target class distribution (bar)
ax1    = fig.add_subplot(gs[0, 0])
counts = df['lung_cancer'].value_counts()
bars   = ax1.bar(counts.index, counts.values,
                 color=[PALETTE['accent'], PALETTE['secondary']],
                 edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 250,
             f'{val:,}\n({val / len(df) * 100:.1f}%)',
             ha='center', va='bottom', fontsize=10,
             fontweight='bold', color=PALETTE['text'])
ax1.set_title(' Target : Lung Cancer Cases', fontweight='bold')
ax1.set_xlabel('Lung Cancer')
ax1.set_ylabel('Patient Count')
ax1.set_facecolor(PALETTE['light_bg'])

# Chart 2 : Age distribution by outcome (histogram)
ax2 = fig.add_subplot(gs[0, 1])
for label, color in [('No Cancer', PALETTE['secondary']),
                     ('Has Cancer', PALETTE['accent'])]:
    val = 'No' if label == 'No Cancer' else 'Yes'
    ax2.hist(df[df['lung_cancer'] == val]['age'],
             bins=30, alpha=0.7, color=color, label=label, edgecolor='white')
ax2.set_title(' Age Distribution by Outcome', fontweight='bold')
ax2.set_xlabel('Age')
ax2.set_ylabel('Count')
ax2.legend(fontsize=9)
ax2.set_facecolor(PALETTE['light_bg'])

# ── Chart 3 : Pack-years smoked by outcome (histogram) ──────
ax3 = fig.add_subplot(gs[0, 2])
for label, color in [('No Cancer', PALETTE['secondary']),
                     ('Has Cancer', PALETTE['accent'])]:
    val = 'No' if label == 'No Cancer' else 'Yes'
    ax3.hist(df[df['lung_cancer'] == val]['pack_years'],
             bins=30, alpha=0.7, color=color, label=label, edgecolor='white')
ax3.set_title(' Pack-Years Smoked by Outcome', fontweight='bold')
ax3.set_xlabel('Pack Years')
ax3.set_ylabel('Count')
ax3.legend(fontsize=9)
ax3.set_facecolor(PALETTE['light_bg'])

# ── Chart 4 : Gender vs Cancer rate (grouped bar) ───────────
ax4      = fig.add_subplot(gs[1, 0])
gender_ct = pd.crosstab(df['gender'], df['lung_cancer'], normalize='index') * 100
gender_ct.plot(kind='bar', ax=ax4,
               color=[PALETTE['secondary'], PALETTE['accent']],
               edgecolor='white', width=0.55)
ax4.set_title(' Gender vs Lung Cancer Rate (%)', fontweight='bold')
ax4.set_xlabel('Gender')
ax4.set_ylabel('Percentage (%)')
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
ax4.legend(['No Cancer', 'Cancer'], fontsize=9)
ax4.set_facecolor(PALETTE['light_bg'])

# ── Chart 5 : Radon exposure vs Cancer rate (grouped bar) ───
ax5      = fig.add_subplot(gs[1, 1])
radon_ct = pd.crosstab(df['radon_exposure'], df['lung_cancer'], normalize='index') * 100
radon_ct.plot(kind='bar', ax=ax5,
              color=[PALETTE['secondary'], PALETTE['accent']],
              edgecolor='white', width=0.55)
ax5.set_title(' Radon Exposure vs Cancer Rate (%)', fontweight='bold')
ax5.set_xlabel('Radon Level')
ax5.set_ylabel('Percentage (%)')
ax5.set_xticklabels(ax5.get_xticklabels(), rotation=0)
ax5.legend(['No Cancer', 'Cancer'], fontsize=9)
ax5.set_facecolor(PALETTE['light_bg'])

# ── Chart 6 : COPD diagnosis vs Cancer rate (grouped bar) ───
ax6     = fig.add_subplot(gs[1, 2])
copd_ct = pd.crosstab(df['copd_diagnosis'], df['lung_cancer'], normalize='index') * 100
copd_ct.plot(kind='bar', ax=ax6,
             color=[PALETTE['secondary'], PALETTE['accent']],
             edgecolor='white', width=0.45)
ax6.set_title(' COPD Diagnosis vs Cancer Rate (%)', fontweight='bold')
ax6.set_xlabel('COPD Diagnosis')
ax6.set_ylabel('Percentage (%)')
ax6.set_xticklabels(ax6.get_xticklabels(), rotation=0)
ax6.legend(['No Cancer', 'Cancer'], fontsize=9)
ax6.set_facecolor(PALETTE['light_bg'])

# ── Chart 7 : Family history vs Cancer rate (grouped bar) ───
ax7   = fig.add_subplot(gs[2, 0])
fh_ct = pd.crosstab(df['family_history'], df['lung_cancer'], normalize='index') * 100
fh_ct.plot(kind='bar', ax=ax7,
           color=[PALETTE['secondary'], PALETTE['accent']],
           edgecolor='white', width=0.45)
ax7.set_title(' Family History vs Cancer Rate (%)', fontweight='bold')
ax7.set_xlabel('Family History')
ax7.set_ylabel('Percentage (%)')
ax7.set_xticklabels(ax7.get_xticklabels(), rotation=0)
ax7.legend(['No Cancer', 'Cancer'], fontsize=9)
ax7.set_facecolor(PALETTE['light_bg'])

# ── Chart 8 : Alcohol consumption vs Cancer rate ─────────────
ax8    = fig.add_subplot(gs[2, 1])
alc_ct = pd.crosstab(df['alcohol_consumption'], df['lung_cancer'], normalize='index') * 100
alc_ct.plot(kind='bar', ax=ax8,
            color=[PALETTE['secondary'], PALETTE['accent']],
            edgecolor='white', width=0.55)
ax8.set_title(' Alcohol Consumption vs Cancer (%)', fontweight='bold')
ax8.set_xlabel('Alcohol Level')
ax8.set_ylabel('Percentage (%)')
ax8.set_xticklabels(ax8.get_xticklabels(), rotation=0)
ax8.legend(['No Cancer', 'Cancer'], fontsize=9)
ax8.set_facecolor(PALETTE['light_bg'])

# ── Chart 9 : Secondhand smoke vs Cancer rate ────────────────
ax9   = fig.add_subplot(gs[2, 2])
ss_ct = pd.crosstab(df['secondhand_smoke_exposure'],
                    df['lung_cancer'], normalize='index') * 100
ss_ct.plot(kind='bar', ax=ax9,
           color=[PALETTE['secondary'], PALETTE['accent']],
           edgecolor='white', width=0.45)
ax9.set_title(' Secondhand Smoke vs Cancer (%)', fontweight='bold')
ax9.set_xlabel('Secondhand Smoke')
ax9.set_ylabel('Percentage (%)')
ax9.set_xticklabels(ax9.get_xticklabels(), rotation=0)
ax9.legend(['No Cancer', 'Cancer'], fontsize=9)
ax9.set_facecolor(PALETTE['light_bg'])

# ── Chart 10 : Asbestos exposure vs Cancer rate ──────────────
ax10   = fig.add_subplot(gs[3, 0])
asb_ct = pd.crosstab(df['asbestos_exposure'],
                     df['lung_cancer'], normalize='index') * 100
asb_ct.plot(kind='bar', ax=ax10,
            color=[PALETTE['secondary'], PALETTE['accent']],
            edgecolor='white', width=0.45)
ax10.set_title(' Asbestos Exposure vs Cancer (%)', fontweight='bold')
ax10.set_xlabel('Asbestos Exposure')
ax10.set_ylabel('Percentage (%)')
ax10.set_xticklabels(ax10.get_xticklabels(), rotation=0)
ax10.legend(['No Cancer', 'Cancer'], fontsize=9)
ax10.set_facecolor(PALETTE['light_bg'])

# ── Chart 11 : Age box-plot by outcome ───────────────────────
ax11 = fig.add_subplot(gs[3, 1])
df.boxplot(column='age', by='lung_cancer', ax=ax11,
           patch_artist=True,
           boxprops     =dict(facecolor=PALETTE['highlight'], color=PALETTE['primary']),
           medianprops  =dict(color=PALETTE['accent'], linewidth=2.5),
           whiskerprops =dict(color=PALETTE['primary']),
           capprops     =dict(color=PALETTE['primary']),
           flierprops   =dict(marker='o', markersize=2,
                              markerfacecolor=PALETTE['secondary'], alpha=0.4))
ax11.set_title(' Age Spread : No vs Yes Cancer', fontweight='bold')
ax11.set_xlabel('Lung Cancer')
ax11.set_ylabel('Age')
plt.sca(ax11); plt.title(''); plt.suptitle('')   # suppress auto-titles
ax11.set_facecolor(PALETTE['light_bg'])

# ── Chart 12 : Pack-years box-plot by outcome ────────────────
ax12 = fig.add_subplot(gs[3, 2])
df.boxplot(column='pack_years', by='lung_cancer', ax=ax12,
           patch_artist=True,
           boxprops     =dict(facecolor=PALETTE['secondary'], color=PALETTE['primary']),
           medianprops  =dict(color=PALETTE['accent'], linewidth=2.5),
           whiskerprops =dict(color=PALETTE['primary']),
           capprops     =dict(color=PALETTE['primary']),
           flierprops   =dict(marker='o', markersize=2,
                              markerfacecolor=PALETTE['highlight'], alpha=0.4))
ax12.set_title(' Pack-Years Spread : No vs Yes Cancer', fontweight='bold')
ax12.set_xlabel('Lung Cancer')
ax12.set_ylabel('Pack Years')
plt.sca(ax12); plt.title(''); plt.suptitle('')
ax12.set_facecolor(PALETTE['light_bg'])

# ── 4. FINAL TITLE & SAVE ───────────────────────────────────
fig.suptitle('Lung Cancer Dataset — Exploratory Data Analysis\n'
             '(50,000 patients | 10 features)',
             fontsize=17, fontweight='bold',
             color=PALETTE['primary'], y=1.01)

plt.savefig('eda_visualizations.png',
            dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
print(" Saved → eda_visualizations.png")

Dataset shape  : 50,000 rows × 11 cols
Missing values : 0
 Saved → eda_visualizations.png


In [40]:
# STEP 5: DEFINE FEATURES & TARGET

# Remove patient_id column
df = df.drop("patient_id", axis=1)

# Define target
target = df.columns[-1]

X = df.drop(target, axis=1)
y = df[target]

In [41]:
# STEP 6: TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [42]:
# STEP 7: FEATURE SCALING

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [43]:
# STEP 8: TRAIN 5 ML MODELS

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

results = {}

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    results[name] = acc

    print("\n==============================")
    print("Model:", name)
    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)



Model: Logistic Regression
Accuracy: 0.7177
Confusion Matrix:
 [[1015 2103]
 [ 720 6162]]

Model: Decision Tree
Accuracy: 0.65
Confusion Matrix:
 [[1419 1699]
 [1801 5081]]

Model: Random Forest
Accuracy: 0.7007
Confusion Matrix:
 [[1385 1733]
 [1260 5622]]

Model: KNN
Accuracy: 0.6986
Confusion Matrix:
 [[1275 1843]
 [1171 5711]]

Model: Naive Bayes
Accuracy: 0.7247
Confusion Matrix:
 [[ 916 2202]
 [ 551 6331]]


In [44]:
# STEP 9: BEST MODEL SELECTION

best_model = max(results, key=results.get)

print("\nBest Model is:", best_model)



Best Model is: Naive Bayes


In [45]:
# STEP 10: PREDICTION EXAMPLES

# Example YES case
example_yes = X_test[0].reshape(1, -1)
print("Prediction (YES case):", model.predict(example_yes))

# Example NO case
example_no = X_test[1].reshape(1, -1)
print("Prediction (NO case):", model.predict(example_no))

Prediction (YES case): [1]
Prediction (NO case): [0]


In [47]:
print("\nEnter details to predict Lung Cancer:\n")

user_data = []

# Now it will NOT ask for patient_id
for col in X.columns:
    val = float(input(f"Enter value for {col}: "))
    user_data.append(val)

# Convert to array
import numpy as np
user_data = np.array(user_data).reshape(1, -1)

# Apply scaling
user_data = scaler.transform(user_data)

# Train best model again (Naive Bayes)
from sklearn.naive_bayes import GaussianNB
best_model = GaussianNB()
best_model.fit(X_train, y_train)

# Prediction
prediction = best_model.predict(user_data)

if prediction[0] == 1:
    print("\nPrediction: YES (High chance of Lung Cancer)")
else:
    print("\nPrediction: NO (Low chance of Lung Cancer)")


Enter details to predict Lung Cancer:

Enter value for age: 50
Enter value for gender: 1
Enter value for pack_years: 12
Enter value for radon_exposure: 1
Enter value for asbestos_exposure: 1
Enter value for secondhand_smoke_exposure: 1
Enter value for copd_diagnosis: 1
Enter value for alcohol_consumption: 1
Enter value for family_history: 1

Prediction: YES (High chance of Lung Cancer)
